# LLM Fine-Tuning Deep Dive, Part 3 of 3: Comparison & Decision

> **Learning objective:** build intuition for how model evaluation matures from “this generated example looks promising” into evidence strong enough to support a workload decision.

Parts 1 and 2 produced the models. Part 3 uses those models as a walking case study in **how to evaluate**. The main story is not which fine-tuning technique came first; it is how each evaluation step answers a weakness in the evidence collected so far.

## The Evaluation Story: Strengthen One Claim at a Time

A model generates a plausible Riverside continuation. What can we honestly conclude?

1. **Observe:** inspect the actual output and describe what appears to have changed.
2. **Question:** turn that impression into a specific claim that could be wrong.
3. **Measure:** hold more of the comparison fixed so the claim is less dependent on one random sample.
4. **Broaden:** check whether the pattern extends beyond the selected example.
5. **Isolate:** determine whether the training choice caused the difference or merely coincided with it.
6. **Decide:** use metrics that match the intended workload, then apply operational release gates.

![Six progressively stronger layers of evaluation evidence from an interesting output to a deployment decision](images/evaluation-evidence-ladder.png)

The left side generates hypotheses; the right side supports decisions. Each layer controls another way the previous evidence could mislead us, but residual risk remains even after a release gate.

Every major section follows the same rhythm:

> **live evidence → a narrower question → a stronger check → a stated limitation → motivation for the next check**

## The Candidate Cast: Two Independent Fine-Tuning Choices

Parts 1 and 2 varied two independent decisions:

- **Behavior objective:** continued pretraining learns domain continuation, SFT learns an instruction contract, and DPO learns relative preferences.
- **Parameter strategy:** full fine-tuning updates every weight, partial freezing updates selected layers, LoRA stores a small low-rank update, and QLoRA reduces frozen-base memory during adapter training.

```mermaid
flowchart LR
    subgraph Behavior["Behavior objective — what should be learned?"]
        direction TB
        B0["Base model<br/>general language"]
        B1["Continued pretraining<br/>domain language and style"]
        B2["SFT<br/>instruction contract"]
        B3["DPO<br/>relative editor preference"]
        B0 --> B1 --> B2 --> B3
    end

    subgraph Parameters["Parameter strategy — where is learning stored?"]
        direction TB
        P0["Full fine-tuning<br/>all weights"]
        P1["Partial freezing<br/>selected layers"]
        P2["LoRA<br/>small adapters"]
        P3["QLoRA<br/>quantized frozen base + adapters"]
        P0 --> P1 --> P2 --> P3
    end
```

The tracks are parallel, not one mandatory pipeline. “SFT + LoRA” means that SFT chooses the behavior to teach while LoRA chooses where the update is stored.

## A Mental Model for Fine-Tuning Evaluation

Evaluation is not a leaderboard and it is not one metric. It is an argument connecting a training intervention to useful behavior under stated constraints:

```mermaid
flowchart LR
    O["Objective<br/>What behavior was taught?"] --> C["Claim<br/>What should improve?"]
    C --> T["Test construct<br/>What observable behavior represents it?"]
    T --> M["Metric + dataset<br/>How is evidence summarized?"]
    M --> D["Decision rule<br/>How much is enough?"]
    D --> R["Residual risk<br/>What can the test miss?"]
```

A broken link invalidates the conclusion. Perplexity is mathematically valid but construct-invalid for an instruction-following claim. Preference win rate fits DPO, but is unconvincing if test prompts or judges leak from preference-data creation. A strong task score is insufficient if latency or safety violates the deployment contract.

### The metric follows the claim, not the technique name

The phrase *each fine-tuning technique needs its own metric* needs one refinement:

- The **training objective** determines the primary behavior metric. Continued pretraining, SFT, and DPO teach different things and require different evidence.
- The **parameter strategy** does not change what success means. Full FT, partial freezing, LoRA, and QLoRA should face the same behavior suite for the same objective, then be compared on efficiency.
- Every candidate also needs **retention checks**. Improving the target while damaging general ability, safety, or an earlier instruction contract is not a clean win.

| Training choice | Claim created by training | Primary evidence | Retention evidence | Efficiency / operational evidence |
| --- | --- | --- | --- | --- |
| Continued pretraining | Riverside prose is modeled better | Clean held-out NLL/perplexity; blinded style or domain-coherence rubric | General-language perplexity; factuality and safety regressions | Training memory/time, checkpoint size, serving latency |
| SFT | The model follows Riverside task contracts | Deterministic task pass rate; format validity; task accuracy or groundedness | Domain behavior, general capability, safety, over-refusal | The same operational measures |
| DPO after SFT | Preferred responses outrank valid alternatives | Blinded pairwise win rate against SFT, with confidence intervals | SFT pass rate, safety, diversity, reward-hacking checks | Preference-data cost plus operational measures |
| Full FT vs partial freeze vs LoRA/QLoRA | The same objective can be learned with a different update budget | **The same primary and retention suite within a matched ablation** | Same suite, split, and seeds | Peak memory, time, artifact size, throughput, latency, cost |

![Objective-specific training signals and the evidence needed for continued pretraining, SFT, and DPO](images/objective-specific-evaluation-evidence.png)

The objective changes the definition of behavioral success. The parameter strategy changes the cost and storage of learning, so it should be compared only after quality is measured with the same objective-specific suite.

A reusable evaluation bundle has five layers:

1. **Target behavior:** did the intended capability improve?
2. **Retention:** what previously working behavior regressed?
3. **Robustness:** does the result survive paraphrases, slices, seeds, and adversarial cases?
4. **Operations:** can it meet memory, latency, throughput, cost, and rollback constraints?
5. **Uncertainty:** is the difference larger than sampling and annotation noise?

The stages below begin with intuition, identify the claim each metric can support, and use code only as a concluding demonstration.

## Table of Concepts and Evaluation Stages

| Phase | Question the reader can now ask | Section |
| ---: | --- | --- |
| 1 | Which model objects and training histories are being compared? | [Reload the six candidates](#setup-reloading-all-six-trained-checkpoints) |
| 2 | Which behavior objective and parameter strategy produced each candidate? | [Separate the two axes](#comparing-the-candidates-without-confusing-the-axes) |
| 3 | What differences appear in live generated outputs? | [Inspect shared-prompt behavior](#side-by-side-every-checkpoint-on-the-same-prompt) |
| 4 | Does a fixed continuation become more or less likely? | [Measure a local probability shift](#demonstration-score-fixed-continuations) |
| 5 | Does the pattern extend across more prose? | [Broaden to a corpus probe](#demonstration-a-shared-corpus-probe-not-a-holdout) |
| 6 | Which metric framework belongs to each objective? | [Separate distribution, task, and preference evidence](#one-model-can-require-several-evaluation-frameworks) |
| 7 | Which combinations were measured, missing, or confounded? | [Map objective × parameter coverage](#technique-combination-grid-data-x-parameter) |
| 8 | What experiment would isolate one training choice? | [Design a controlled ablation](#ablation-study-what-happens-if-you-skip-a-stage) |
| 9 | What has and has not been established? | [Checkpoint the evidence](#what-this-fine-tuning-arc-established) |
| 10 | Which candidate belongs to each Riverside workload? | [Make the workload decision](#the-decision-what-can-riverside-hand-off-today) |
| 11 | Can the reader repeat the reasoning process on one scenario? | [Run the guided walking example](#guided-walking-example-one-scenario-through-the-evidence-ladder) |
| 12 | What must pass before deployment? | [Apply production gates](#from-notebook-shortlist-to-production-control-plane) |

---

## Setup: Reloading All Six Trained Checkpoints

Parts 1 and 2 ran in separate kernels and saved candidates under `./checkpoints/`, so this notebook reloads fresh Python objects rather than relying on hidden state.

> **Prerequisite:** Rerun Parts 1 and 2 from clean kernels with `HuggingFaceTB/SmolLM2-135M-Instruct` before running this notebook. Artifacts from other architectures cannot be reloaded here.

The six candidates are evidence sources, not one chronological checkpoint chain. Some differ in objective, parameter strategy, corpus, and hyperparameters; Part 3 will expose those confounds rather than treating the resulting scores as a causal ranking.

### Candidate Manifest: What Will Be Reloaded?

The notebook needs six independent model objects so that loading one adapter cannot mutate another candidate's base model.

| Candidate | Saved artifact | Objective | Parameter strategy | Ancestry |
| --- | --- | --- | --- | --- |
| Baseline | Hugging Face base checkpoint | Original pretraining | No Riverside update | SmolLM2 base |
| Full-FT continuation | `non-instruction-full` | Continued pretraining | Full fine-tuning | SmolLM2 base |
| Partial-freeze continuation | `partial-freeze` | Continued pretraining | Selected late layers | SmolLM2 base |
| LoRA continuation | `peft-lora` | Continued pretraining | LoRA | Fresh SmolLM2 base + adapter |
| SFT assistant | `instruction-lora` | SFT | LoRA | Fresh SmolLM2 base + adapter |
| DPO assistant | `preference-dpo` | DPO | Continue SFT LoRA | Fresh SmolLM2 base + DPO adapter |

The code first restores the common tokenizer and prompt contract, then loads a fresh base for every PEFT adapter, verifies the LoRA target modules, and derives parameter counts from the objects actually loaded. This reconstructs the experiment before any comparison begins.

> **PyTorch → Keras:** `torch.cuda.is_available()` + `.to(device)` explicitly move a model/tensors to
> GPU or CPU, `AutoModelForCausalLM.from_pretrained(...)` loads pretrained weights, and
> `model.generate(...)` run inside `torch.no_grad()` performs autoregressive decoding without tracking
> gradients (nothing to backprop through during inference). **Keras/TF equivalent:** TensorFlow places
> ops on GPU automatically (explicit placement is `tf.device(...)`, rarely needed); the loading call
> would be `TFAutoModelForCausalLM.from_pretrained(...)` followed by the same `.generate(...)` method --
> Keras/TF has no separate "no_grad" context since inference doesn't build a gradient tape by default.


In [ ]:
# Re-establish Parts 1-2's foundations using the same SmolLM2 base and prompt contract.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
SYSTEM_PROMPT = "You are a careful fiction-writing assistant for Riverside Publishing."
CONTINUATION_INSTRUCTION = "Continue the fiction narrative in the same style."

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"

num_hidden_layers = base_model.config.num_hidden_layers
hidden_size = base_model.config.hidden_size
total_base_parameters = sum(parameter.numel() for parameter in base_model.parameters())
print(
    f"Loaded {MODEL_NAME}: {total_base_parameters:,} parameters, "
    f"{num_hidden_layers} decoder layers, hidden size {hidden_size}."
)


def instruction_prompt(prompt):
    """Build the user message used by the SFT and DPO recipes in Parts 1-2."""
    return f"{CONTINUATION_INSTRUCTION}\n\n{prompt}"


def apply_instruction_template(prompt):
    """Serialize an instruction through the model's native chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate(model, prompt, max_new_tokens=60, use_chat_template=False):
    """Generate only new tokens, using the native chat format for instruction candidates."""
    model.eval()
    model_input = apply_instruction_template(prompt) if use_chat_template else prompt
    inputs = tokenizer(model_input, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        output_ids[0][prompt_length:], skip_special_tokens=True
    ).strip()
    return completion if completion else "[model stopped immediately after the prompt]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")


### Reloading the Five Fine-Tuned Checkpoints

Each PEFT-wrapped adapter (instruction-tuned LoRA, DPO, LoRA continued pretraining) gets its own fresh
base-model instance rather than sharing `base_model` above -- the same "every PEFT wrapper gets its
own base" rule Parts 1-2 followed throughout. `freeze_model`'s `requires_grad` flags are re-applied
after loading (see the comment below) since that bookkeeping isn't part of a saved checkpoint -- only
the trained weights are.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, path)` wraps a fresh base model with a
> saved LoRA adapter's weights; `named_parameters()` iterates `(name, tensor)` pairs so `requires_grad`
> can be toggled per-parameter (used here to re-apply the freeze pattern, since that bookkeeping isn't
> part of a saved checkpoint), and `p.numel()` counts a tensor's elements to total trainable params.
> **Keras/TF equivalent:** LoRA loading has no single standard TF API (usually a custom `tf.keras.Model`
> subclass or a TF-specific PEFT integration); freezing is coarser-grained -- `layer.trainable = False`
> per layer rather than per-parameter -- and element counts come from `tf.size(variable)`.


In [ ]:
# Reload only artifacts regenerated by Parts 1-2 for MODEL_NAME.
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, "./checkpoints/instruction-lora"
).to(device)

dpo_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model = PeftModel.from_pretrained(
    dpo_base_reload, "./checkpoints/preference-dpo"
).to(device)

freeze_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/partial-freeze"
).to(device)
n_layers = freeze_model.config.num_hidden_layers
unfreeze_from = n_layers - max(2, n_layers // 4)

for parameter in freeze_model.parameters():
    parameter.requires_grad = False
for layer in freeze_model.model.layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True
for parameter in freeze_model.lm_head.parameters():
    parameter.requires_grad = True

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = PeftModel.from_pretrained(
    lora_pt_base, "./checkpoints/peft-lora"
).to(device)

adapter_models = {
    "instruction LoRA": instruct_lora_model,
    "DPO policy": policy_model,
    "continued-pretraining LoRA": lora_pt_model,
}
expected_targets = set(LORA_TARGET_MODULES)
for adapter_name, adapter_model in adapter_models.items():
    configured_targets = {
        target
        for peft_config in adapter_model.peft_config.values()
        for target in peft_config.target_modules
    }
    if configured_targets != expected_targets:
        raise ValueError(
            f"{adapter_name} targets {sorted(configured_targets)}, expected "
            f"{sorted(expected_targets)}. Rerun Parts 1-2 with the SmolLM2 LoRA recipe."
        )

for model in (
    non_instruct_ckpt,
    instruct_lora_model,
    policy_model,
    freeze_model,
    lora_pt_model,
):
    model.eval()

# Verify that the reconstructed partial-freeze model matches the intended layer policy.
trainable_partial_names = [
    name for name, parameter in freeze_model.named_parameters() if parameter.requires_grad
]
assert any(
    name.startswith(f"model.layers.{unfreeze_from}.")
    for name in trainable_partial_names
), "Expected SmolLM2 trailing-layer parameter names were not found"

print("Reloaded all six candidates (baseline + 5 fine-tuned).")
print(f"SmolLM2 layers/hidden size: {n_layers} / {freeze_model.config.hidden_size}")


---

## Comparing the Candidates Without Confusing the Axes

Before comparing outputs, place every candidate on the two tracks introduced above:

| Candidate | Behavior objective | Parameter strategy | Limitation it was created to address |
| --- | --- | --- | --- |
| Baseline | Original pretraining only | No Riverside update | Control: capable language model, but no Riverside adaptation |
| Continued pretraining | Next-token prediction on Riverside prose | Full fine-tuning | Learn catalog language and house style |
| Partial-freeze continuation | Next-token prediction on Riverside prose | Update selected late layers | Test whether less trainable state can carry domain adaptation |
| LoRA continuation | Next-token prediction on Riverside prose | Low-rank adapters | Store domain adaptation in a small swappable artifact |
| SFT LoRA | Supervised prompt/completion loss | Low-rank adapters | Teach direct instruction following and response format |
| DPO adapter | Chosen/rejected preference loss after SFT | Continue updating the SFT adapter | Prefer editor-ranked responses among plausible answers |

A name such as **SFT + LoRA** contains two independent decisions:

- **SFT** says what behavior the examples and loss teach.
- **LoRA** says where the resulting update is stored.

### Two comparisons that answer different questions

**Behavior progression:** baseline → continued pretraining → SFT → DPO asks whether each objective addresses a new workload limitation. These checkpoints are not interchangeable: prose perplexity cannot determine whether an assistant follows instructions, and instruction pass rate cannot establish editor preference.

**Parameter progression:** full fine-tuning → partial freezing → LoRA asks how much model state must change to learn the **same** behavior. This requires a controlled parameter ablation: same base model, data split, ordered examples, optimizer policy, token budget, seeds, and evaluation suite; only the update strategy changes.

The current full, partial-freeze, and LoRA runs are useful demonstrations, but they are **not** that controlled ablation because their data and hyperparameters differ. Any score gap mixes parameter strategy with those other differences.

### What is absent from the experiment?

The complete design space has three objectives by three parameter strategies. Five combinations were trained. SFT with full fine-tuning or partial freezing and DPO with full fine-tuning or partial freezing remain unmeasured. QLoRA was explored as a memory-scaling mechanism in Part 2, not trained as another quality candidate.

Blank combinations mean **not measured**, not failed.

## Stage 1 — Observe Behavior Before Scoring It

### The walking problem: complete one Riverside sentence

One sentence will carry the evaluation story from sample generation to release reasoning:

| Role | Text |
| --- | --- |
| Shared prompt | `Aria Voss checked the Meridian's Promise status panel and` |
| Riverside continuation to score | ` opened the Keeper's maintenance logs` |
| Generic control continuation | ` looked at the screen` |

The working hypothesis is deliberately narrow:

> Continued pretraining made the Riverside continuation less surprising than it was to the base model, and changed it more selectively than the generic control.

Each stage strengthens or limits that claim:

1. **Observe:** do generated completions look more Riverside-specific?
2. **Inspect the mechanism:** at each position, how much probability does each model assign to the actual next token in the fixed continuation?
3. **Summarize locally:** did the continuation's mean log-probability improve relative to the control?
4. **Broaden:** does lower surprise extend to many frozen Riverside passages, producing lower held-out perplexity?
5. **Diagnose:** were model, data, and training conditions controlled well enough to attribute the difference to continued pretraining?
6. **Choose the right evidence:** would this distribution metric answer Riverside's deployment claim, or does the workload require task, preference, safety, and operational evidence instead?

Before choosing a metric, inspect what changed. Generated examples are useful because failures are concrete: the model may use generic prose, ignore an instruction, violate a format, invent a catalog fact, or stop badly. These observations reveal the **construct** the later benchmark must measure.

But examples answer only *what can happen*, not *how often it happens*. Generation is affected by prompt wording, chat formatting, temperature, seed, and output length. A compelling sample is therefore a hypothesis generator, not a score.

Read each output through three lenses:

1. **Domain behavior:** does it use Riverside-specific entities and relationships coherently rather than merely echoing a name?
2. **Task behavior:** does it continue prose or answer an instruction in the requested format and stop appropriately?
3. **Preference behavior:** does DPO differ consistently from SFT, or is one nicer-sounding sample just decoding randomness?

> **Predict:** Which failure will be easiest to see but hardest to quantify: generic style, instruction non-compliance, or editorial preference? Record one observable sign before running the comparison.

The next code cell is a demonstration of observation, not an evaluation harness. One sampled generation cannot support a ranking. That limitation motivates Stage 2: hold the walking problem's two continuations fixed and inspect how training changed their token probabilities.

### Side-by-Side: Every Checkpoint on the Same Prompt

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained("./checkpoints/...")` reloads a saved
> fine-tuned checkpoint from disk into a fresh `torch.nn.Module`, then `.to(device)` places it on
> GPU/CPU before the loop below calls the `generate()` helper defined earlier on each model in turn.
> **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(path)` loads the same checkpoint
> format into a `tf.keras.Model`; TensorFlow doesn't need an explicit `.to(device)` call since device
> placement is handled by default device scoping (or `tf.distribute` for multi-device setups) instead.


In [ ]:
# Compare every candidate on shared catalog prompts.
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("QUALITATIVE CANDIDATE EXAMPLES - SAME PROMPTS, ONE SAMPLE EACH")
print("=" * 80)

for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f'\nPrompt ({prompt_name}): "{prompt}"')
    print("-" * 80)

    for model_index, (model_name, model) in enumerate(models_to_test.items()):
        sample_seed = 42 + model_index
        torch.manual_seed(sample_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(sample_seed)

        uses_chat_template = "Instruction" in model_name or "Preference" in model_name
        effective_prompt = instruction_prompt(prompt) if uses_chat_template else prompt
        output = generate(
            model,
            effective_prompt,
            max_new_tokens=60,
            use_chat_template=uses_chat_template,
        )
        output_display = output[:120] + "..." if len(output) > 120 else output

        format_note = " + SmolLM2 chat template" if uses_chat_template else ""
        print(f"\n[{model_name}] seed={sample_seed}{format_note}")
        print(f"  Output: {output_display}")

print("\n" + "=" * 80)
print("READ THESE AS EXAMPLES:")
print("1. Catalog language: are names and setting details specific rather than generic?")
print("2. Task behavior: does the model continue prose or answer an instruction?")
print("3. Stability: a claim requires repeated prompts/samples and a scoring rubric.")
print("=" * 80)

### Stage 1B — Inspect One Complete Output

The prompt matrix gives breadth but truncates each sample. Before leaving qualitative evidence, inspect one complete output from every candidate and expose the exact input format.

This matters because SFT and DPO learned through SmolLM2's chat template, while continuation models learned from plain prose. Sending every model the same raw string would test prompt mismatch as well as model behavior.

Use the same three lenses:

| Lens | Concrete sign to look for | Common false positive |
| --- | --- | --- |
| Domain language | Story-specific entities or relationships used coherently | Repeating a name copied from the prompt |
| Task behavior | Direct answer or bounded continuation in the requested format | Fluent prose that ignores the instruction |
| Preference signal | A repeatable difference between SFT and DPO | One nicer sample caused by decoding randomness |

Even the complete outputs remain sampled observations. Stage 2 removes decoding randomness by scoring the same fixed phrases under both models.

In [ ]:
# Instruction and preference candidates use the chat contract from Parts 1-2.
instruct_prompt = instruction_prompt(PROMPT)
print(f"Shared prompt (plain models)       : {PROMPT!r}")
print(f"Shared user request (chat models) : {instruct_prompt!r}")
print()

print("=== Baseline (no fine-tuning) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(base_model, PROMPT)}")
print()

print("=== Non-instructional continued pretraining (full fine-tune) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(non_instruct_ckpt, PROMPT)}")
print()

print("=== Instruction-tuned (LoRA) ===")
print(f"  Input : SmolLM2 chat template over {instruct_prompt!r}")
print(
    f"  Output: {generate(instruct_lora_model, instruct_prompt, use_chat_template=True)}"
)
print()

print("=== Preference-aligned (DPO on the instruction-tuned adapter) ===")
print(f"  Input : SmolLM2 chat template over {instruct_prompt!r}")
print(f"  Output: {generate(policy_model, instruct_prompt, use_chat_template=True)}")
print()

print("=== Partial fine-tuning (layer freezing) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(freeze_model, PROMPT)}")
print()

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(lora_pt_model, PROMPT)}")
print()

## Stage 2 — Hold the Text Fixed and Measure Probability Shift

**Question inherited from Stage 1:** was an apparent improvement caused by changed model probabilities or by one lucky decoding path?

We now freeze the walking problem:

- prompt: `Aria Voss checked the Meridian's Promise status panel and`
- Riverside target: ` opened the Keeper's maintenance logs`
- generic control: ` looked at the screen`

### Foundation: what is a next-token probability space?

A language model does not produce the target continuation in one step. It repeats one operation: given all tokens seen so far, assign a score to **every token in its vocabulary** as the possible next token.

1. The tokenizer converts text into token IDs. A token may be a word, part of a word, punctuation, or whitespace-bearing fragment.
2. The model reads the current token sequence and emits one raw score, called a **logit**, for each vocabulary token.
3. Softmax converts those logits into nonnegative probabilities that sum to $1$.
4. One probability is attached to each possible next token. That complete vector is the **next-token probability distribution** for this exact context.

Immediately after the walking prompt, an illustrative distribution might look like this:

| Candidate next token | Conditional probability |
| --- | ---: |
| ` opened` | $0.50$ |
| ` looked` | $0.20$ |
| ` said` | $0.10$ |
| every other vocabulary token combined | $0.20$ |

These are alternatives at **one position**, so they sum to $1$. If ` opened` is appended, the context changes and the model computes a new distribution for the next position, perhaps assigning probability $0.20$ to ` the`. “Probability space” here is not a mysterious hidden geometry; it is the vocabulary alternatives and their context-dependent probability mass at each prediction step.

A high probability means “this token is expected under the model after this context.” It does not mean the token is true, safe, or best for the user.

### Generation versus scoring

A generated paragraph combines two systems:

```mermaid
flowchart LR
    P["Walking prompt tokens"] --> M["Model<br/>one probability per vocabulary token"]
    M --> S["Decoder chooses one token<br/>temperature, top-p, seed"]
    S --> C["Append chosen token<br/>build a new context"]
    C --> M
    S --> O["Eventually: one sampled completion"]
    M --> F["Alternative: gather each known target token<br/>to score the fixed continuation"]
```

The model supplies a distribution; the decoder selects from it. Greedy decoding picks the largest probability, while sampling can pick another token according to a temperature/top-$p$ policy. A new seed can therefore change the completion without changing the model's probabilities at all.

To compare models more directly, keep the prompt and target fixed. At each position, look up the probability assigned to the token that actually appears next. This is **teacher forcing**: when scoring token $w_t$, provide the real preceding target tokens $w_{<t}$ rather than feeding back a model-generated path. Every model is tested on exactly the same contexts and targets.

### Follow the walking continuation token by token

To keep the arithmetic readable, consider the first two illustrative target tokens. The real tokenizer may divide the text differently; the next code cell prints its actual tokenization and scores every resulting token.

| Scoring step | Context supplied to model | Actual target token | Probability gathered |
| ---: | --- | --- | ---: |
| 1 | `... status panel and` | ` opened` | $0.50$ |
| 2 | `... status panel and opened` | ` the` | $0.20$ |

The probability assigned to this two-token prefix is the product of its conditional probabilities:

$$
p(\text{ opened the}\mid\text{walking prompt})
=0.50\times0.20=0.10.
$$

The full target adds probabilities for `Keeper`, `'s`, `maintenance`, `logs`, or whatever token pieces the tokenizer actually produces. Products across many tokens quickly become tiny. Logs turn multiplication into addition:

$$
\log 0.50+\log 0.20=-0.693-1.609=-2.303.
$$

Divide by the two predicted tokens to place this prefix on a per-token scale:

$$
\text{mean log-probability}
=\frac{-2.303}{2}=-1.151\ \text{nats/token}.
$$

For the complete $T$-token target, the same calculation is

$$
\frac{1}{T}\sum_{t=1}^{T}\log p(w_t\mid\text{walking prompt},w_{<t}).
$$

Because probabilities are at most $1$, their natural logs are zero or negative. A **higher** value, such as $-1.2$ instead of $-4.7$, means the fixed target tokens were less surprising. Negative log-likelihood (NLL) flips the sign, so **lower NLL is better**.

The absolute value is not a universal quality grade. It depends on the tokenizer, fixed text, and serialized context. A score shift is interpretable only when candidates use the same tokenizer and receive the same token sequence.

### Which fine-tuning claim can this test?

This is a natural mechanism check for **continued pretraining** because its training objective directly changes next-token likelihood over prose. It is not the primary metric for SFT or DPO:

| Objective | What fixed-text probability can reveal | What it cannot establish |
| --- | --- | --- |
| Continued pretraining | Whether the Riverside target became less surprising than before, especially relative to the generic control | Overall style quality, factual correctness, or generalization beyond selected text |
| SFT | Whether a particular reference answer became more likely | Whether the model follows varied instructions or validly solves the task |
| DPO | Whether chosen and rejected responses moved relative to the SFT reference | Whether humans prefer newly generated responses across held-out prompts |

For SFT, the stronger construct is success across a deterministic instruction suite. For DPO, it is blinded preference win rate against the SFT reference while retaining instruction ability.

### Prediction before measurement

For the walking prompt, continued pretraining should increase the score of ` opened the Keeper's maintenance logs` more than the generic control ` looked at the screen`. The control matters: if both rise equally, the change is broad rather than Riverside-selective.

> **Predict:** If the Riverside target improves by $+0.8$ nats/token but the generic control also improves by $+0.8$, what part of the walking hypothesis remains unsupported?

### Demonstration: score fixed continuations

The next code cell performs exactly the teacher-forced lookup above for every actual continuation token. It first prints a token-level trace for the walking target, then compares complete phrase scores. This removes decoding randomness and tests one mechanism. It does **not** establish that generated claims are correct, instructions are followed, or the effect extends across the corpus.

That remaining coverage problem motivates Stage 3: ask whether the same lower-surprise pattern holds across many unseen Riverside examples. [Calibration and Confidence Evaluation](../05-llm-evaluation/04-calibration-and-confidence.ipynb) later explains why expected-token probability is not calibrated correctness.

> **PyTorch → Keras:** `model.eval()` switches dropout/batch-norm-style layers to inference behavior;
> `torch.no_grad()` disables gradient tracking for the forward pass below; calling `model(**inputs)`
> runs a forward pass and returns `outputs.logits` (raw scores), and `F.softmax(logits, dim=-1)`
> (from `torch.nn.functional`) converts those logits into a probability distribution over the vocabulary.
> **Keras/TF equivalent:** Keras layers infer train/inference behavior automatically (or via a
> `training=False` argument) instead of an explicit `.eval()` call, and there's no separate "no_grad"
> context since plain forward calls outside a `GradientTape` don't track gradients; the softmax step is
> `tf.nn.softmax(logits, axis=-1)` -- same idea, `axis` instead of `dim`.


In [ ]:
# Trace one fixed sentence token by token, then compare complete continuation scores.
import math

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

prompt_for_analysis = "Aria Voss checked the Meridian's Promise status panel and"
walking_target_label = "Keeper's maintenance logs"

# The first phrase is the walking target; the remaining phrases test selectivity.
candidate_phrases = {
    walking_target_label: " opened the Keeper's maintenance logs",
    "quantum fold drive": " checked the quantum fold drive",
    "containment-field anomaly": " detected a containment-field anomaly",
    "looked at the screen": " looked at the screen",
    "said nothing": " said nothing",
    "went back to work": " went back to work",
}
domain_labels = set(list(candidate_phrases)[:3])


def continuation_logprob(model, prompt, continuation):
    """Return sequence summaries and the actual-token trace for one continuation."""
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = tokenizer(
        prompt + continuation, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + continuation")

    model.eval()
    with torch.no_grad():
        logits = model(full_ids).logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    # Position prompt_length-1 predicts the first continuation token.
    continuation_ids = full_ids[:, prompt_length:]
    continuation_log_probs = log_probs[
        0, prompt_length - 1 : full_ids.shape[1] - 1
    ].gather(1, continuation_ids[0].unsqueeze(1)).squeeze(1)

    token_ids = continuation_ids[0].tolist()
    token_log_probs = continuation_log_probs.tolist()
    token_pieces = tokenizer.convert_ids_to_tokens(token_ids)
    trace = [
        {
            "token_id": token_id,
            "token": token_piece,
            "probability": math.exp(token_log_probability),
            "log_probability": token_log_probability,
            "surprise": -token_log_probability,
        }
        for token_id, token_piece, token_log_probability in zip(
            token_ids, token_pieces, token_log_probs
        )
    ]

    return {
        "mean": continuation_log_probs.mean().item(),
        "sum": continuation_log_probs.sum().item(),
        "tokens": continuation_ids.shape[1],
        "trace": trace,
    }


phrase_results = []
walking_traces = None
for label, phrase in candidate_phrases.items():
    baseline_score = continuation_logprob(base_model, prompt_for_analysis, phrase)
    finetuned_score = continuation_logprob(
        non_instruct_ckpt, prompt_for_analysis, phrase
    )
    if label == walking_target_label:
        walking_traces = {
            "baseline": baseline_score["trace"],
            "finetuned": finetuned_score["trace"],
        }
    phrase_results.append(
        {
            "label": label,
            "kind": "catalog" if label in domain_labels else "generic control",
            "tokens": finetuned_score["tokens"],
            "baseline": baseline_score["mean"],
            "finetuned": finetuned_score["mean"],
            "delta": finetuned_score["mean"] - baseline_score["mean"],
        }
    )

assert walking_traces is not None
assert [item["token_id"] for item in walking_traces["baseline"]] == [
    item["token_id"] for item in walking_traces["finetuned"]
]

print("=== Walking problem: actual next-token trace ===")
print(f"Prompt: {prompt_for_analysis!r}")
print(f"Fixed target: {candidate_phrases[walking_target_label]!r}\n")
print(
    f"{'Step':>4} {'Tokenizer piece':22} {'Base p':>10} {'Adapted p':>10} "
    f"{'Base surprise':>14} {'Adapted surprise':>17}"
)
print("-" * 92)
for step, (baseline_token, finetuned_token) in enumerate(
    zip(walking_traces["baseline"], walking_traces["finetuned"]), start=1
):
    print(
        f"{step:>4} {baseline_token['token']!r:22} "
        f"{baseline_token['probability']:>10.5f} "
        f"{finetuned_token['probability']:>10.5f} "
        f"{baseline_token['surprise']:>14.3f} "
        f"{finetuned_token['surprise']:>17.3f}"
    )
print(
    "\nRead one row at a time: both models see the same true prefix; lower surprise "
    "means the model assigned more probability to that actual next token."
)

print("\n=== Complete fixed-continuation scores ===")
print(f"{'Phrase':30} {'Type':16} {'Tok':>3} {'Base':>9} {'Fine-tuned':>11} {'Delta':>9}")
print("-" * 86)
for row in phrase_results:
    print(
        f"{row['label']:30} {row['kind']:16} {row['tokens']:>3} "
        f"{row['baseline']:>9.3f} {row['finetuned']:>11.3f} {row['delta']:>+9.3f}"
    )

labels = [row["label"] for row in phrase_results]
baseline_values = [row["baseline"] for row in phrase_results]
finetuned_values = [row["finetuned"] for row in phrase_results]
deltas = [row["delta"] for row in phrase_results]
positions = np.arange(len(labels))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.35, 1]})

axes[0].barh(
    positions - width / 2,
    baseline_values,
    height=width,
    label="Baseline",
    color="#4C78A8",
)
axes[0].barh(
    positions + width / 2,
    finetuned_values,
    height=width,
    label="Continued pretraining",
    color="#E45756",
)
axes[0].set_yticks(positions)
axes[0].set_yticklabels(labels)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean log-probability per token (higher = less surprising)")
axes[0].set_title("Walking Target and Controls: Fixed-Text Scores")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.25)

delta_colors = ["#2A9D8F" if value >= 0 else "#D1495B" for value in deltas]
axes[1].barh(positions, deltas, color=delta_colors)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Change after continued pretraining (nats/token)")
axes[1].set_title("Probability Shift, Without a Quality Claim")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

catalog_deltas = [row["delta"] for row in phrase_results if row["kind"] == "catalog"]
control_deltas = [
    row["delta"] for row in phrase_results if row["kind"] == "generic control"
]
walking_delta = next(
    row["delta"] for row in phrase_results if row["label"] == walking_target_label
)
print(
    f"\nWalking-target shift:     {walking_delta:+.3f} nats/token\n"
    f"Mean catalog shift:       {np.mean(catalog_deltas):+.3f} nats/token\n"
    f"Mean generic-control shift: {np.mean(control_deltas):+.3f} nats/token"
)
print(
    "Interpretation: positive shifts mean these fixed phrases became less surprising. "
    "A Riverside-specific claim additionally requires catalog shifts to exceed controls."
)

## Stage 3 — Broaden from the Walking Sentence to Corpus Fit

**Question inherited from Stage 2:** did the Riverside target become less surprising only because we selected it, or does the pattern extend across Riverside prose?

### Turn the same token trace into average surprise

Stage 2 printed each actual target token and its probability. Perplexity does not inspect a different model mechanism; it summarizes the same teacher-forced next-token scoring over more tokens.

Return to the illustrative first two positions of ` opened the Keeper's maintenance logs`:

| Actual target token | Probability assigned by model | Token surprise $-\log p$ |
| --- | ---: | ---: |
| ` opened` | $0.50$ | $0.693$ |
| ` the` | $0.20$ | $1.609$ |
| **Total / average** | | $2.302 / 2 = 1.151$ |

Negative log-probability is called **surprise** because an expected token contributes little penalty while an unlikely observed token contributes a large one:

| Probability of actual next token | Surprise $-\log p$ |
| ---: | ---: |
| $1.00$ | $0.000$ |
| $0.50$ | $0.693$ |
| $0.10$ | $2.303$ |
| $0.01$ | $4.605$ |

For the full walking target, repeat this for every actual tokenizer piece, sum the surprises, and divide by the number of target tokens. That produces mean negative log-likelihood, or mean NLL. In language modeling this is also token-level cross-entropy against the observed text.

### Why exponentiate?

Mean NLL is mathematically convenient but expressed in log units. Exponentiation returns it to a probability-like scale:

$$
\operatorname{PPL}
=
\exp\left(
-\frac{1}{N}\sum_{t=1}^{N}\log p(w_t\mid w_{<t})
\right).
$$

For the two illustrative walking-target positions,

$$
\operatorname{PPL}
=
\exp(1.151)
=
\left(\frac{1}{0.50\times0.20}\right)^{1/2}
\approx 3.16.
$$

The middle expression reveals what perplexity is: the **geometric mean of the inverse probability assigned to each actual token**. Lower is better:

- Perplexity $1$ is the theoretical floor: the model assigned probability $1$ to every observed next token.
- Perplexity $3.16$ here summarizes the combined $0.50$ and $0.20$ token probabilities.
- Increasing perplexity means the observed text was, on average, more surprising to the model.

The common “effective choice count” analogy is now easier to place. If the model assigned probability $1/k$ to the actual token at every position, perplexity would be $k$. Thus perplexity $20$ is *as surprising on average* as repeatedly finding the answer among 20 equally likely choices. The real distribution is not uniform, and the alternatives change with context, so perplexity is not a literal count of tokens considered.

### One sentence is still selected evidence

Suppose the adapted model assigns lower surprise than the base model to most tokens in ` opened the Keeper's maintenance logs`, and its mean log-probability shift exceeds the shift for ` looked at the screen`. That supports the walking hypothesis **for these fixed continuations**.

It still leaves a selection problem: we chose a target that sounds Riverside-specific. A credible distribution claim must preserve the operation but replace the selected sentence with a frozen collection of unseen Riverside text.

```mermaid
flowchart LR
    A["Walking target<br/>selected fixed continuation"] --> B["Token trace<br/>probability of each actual token"]
    B --> C["Local mean NLL / perplexity<br/>mechanism evidence"]
    C --> D["Frozen evaluation corpus<br/>many unseen passages"]
    D --> E["Token-weighted corpus perplexity<br/>distribution evidence"]
    E --> F["Slices + retention set<br/>where did fit improve or regress?"]
```

The computation does not change:

1. tokenize frozen evaluation text into observed targets;
2. teacher force each position and gather the actual-token probability;
3. convert each probability to surprise $-\log p$;
4. sum surprise over all evaluated tokens and divide by token count;
5. exponentiate once to obtain corpus perplexity.

Token weighting matters. Averaging paragraph-level losses would give a short paragraph the same influence as a long one. Summing NLL over all predicted tokens and dividing once by the total token count makes each evaluated token contribute equally.

### The dataset is part of the metric

“Perplexity = 18” is incomplete. The evidence is **perplexity of model M, with tokenizer T, on dataset split D, under context policy P**. Values are not directly comparable across different tokenizers because the units being predicted changed.

A credible continued-pretraining evaluation therefore needs:

1. A file-level test split frozen before training, with duplicate and near-duplicate checks.
2. The same tokenizer, truncation policy, context length, and token aggregation for every candidate.
3. Slices that reveal where gains occur instead of only one corpus-wide average.
4. An out-of-domain retention set to detect catastrophic forgetting.
5. Repeated training seeds or bootstrap intervals when a small score difference will drive a decision.

### Where perplexity belongs in the evaluation framework

| Claim | Perplexity's role | Required companion evidence |
| --- | --- | --- |
| Continued pretraining improved domain modeling | **Primary intrinsic metric** on a clean held-out domain split | Blinded style/coherence rubric, factual checks, general-domain retention |
| SFT improved instruction following | Diagnostic only; reference answers may have lower loss without reliable task compliance | Deterministic task pass rate, format validity, task accuracy/groundedness |
| DPO improved preference alignment | Diagnostic only; DPO optimizes chosen versus rejected responses relatively | Blinded win rate versus SFT, confidence interval, SFT-retention suite |
| LoRA matched full FT for one objective | Use the same perplexity or task suite for both under a matched protocol | Memory, time, artifact size, latency, throughput, cost |

Perplexity measures fit to observed text. It cannot tell whether prose is factually correct, whether an answer follows an instruction, whether editors prefer it, or whether it is safe. Low perplexity can even reward confidently reproducing false or contaminated text.

### Demonstration: a shared corpus probe, not a holdout

The next code cells apply the walking problem's aggregation to later-chapter paragraphs and score every candidate on the same sampled text. This improves coverage, but the upstream runs did not use one predeclared file-level split:

- Full fine-tuning saw some chapters sampled by this probe.
- Partial freezing and LoRA used smaller and different chapter subsets.
- SFT and DPO used different data formats and coverage.

Therefore this is a **shared corpus probe**, not clean held-out evaluation. Its ranking combines training exposure, behavior objective, parameter strategy, and hyperparameters.

| What the demonstration improves | What remains unresolved |
| --- | --- |
| Extends the same next-token scoring from one selected sentence to many tokens | Some evaluated text overlaps training exposure |
| Sends every candidate the same paragraphs | Candidates learned from different corpora and objectives |
| Aggregates loss by token rather than paragraph | A lower value cannot identify which training choice caused it |

> **Predict:** Which candidate might achieve low Riverside prose perplexity yet be the wrong editing assistant, and why?

The code broadens the evidence, but the lowest bar must not be called the best model. That unresolved attribution and construct problem motivates Stage 4: identify which objective each metric can support and which comparisons remain confounded.

[LLM Evaluation, Part 1](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb#part-5--perplexity-how-surprised-is-the-model-by-its-own-output) develops perplexity further.

> **PyTorch → Keras:** passing `labels=enc["input_ids"]` into the model's forward call makes the
> Hugging Face model compute cross-entropy loss internally and return it as `out.loss` (a scalar
> tensor); `torch.no_grad()` skips gradient tracking since this is evaluation-only, and `.item()`
> pulls the plain Python float out of that 0-d tensor, which `math.exp(loss)` then turns into
> perplexity. **Keras/TF equivalent:** the TF counterpart model supports the same `labels=` convenience
> (`model(enc, labels=...)`), while plain Keras code would instead call
> `tf.keras.losses.SparseCategoricalCrossentropy()(y_true, logits)` and use `.numpy()` in place of
> `.item()` to extract the scalar.


In [ ]:
# Resolve the Riverside corpus only when the corpus-level metric first needs it.
try:
    notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        notebook_dir = Path(__file__).parent
    except NameError:
        notebook_dir = Path.cwd()

CONTENT_DIR = notebook_dir / "content"
if not CONTENT_DIR.exists():
    fallback_content_dir = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if fallback_content_dir.exists():
        CONTENT_DIR = fallback_content_dir

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}

if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Riverside content directory not found at {CONTENT_DIR.absolute()}. "
        "Run this notebook from the repository workspace."
    )

print(f"Corpus evaluation setup: {CONTENT_DIR.absolute()} ({len(NOVELS)} novels)")

In [ ]:
import math


# Use the same later-chapter sample for every candidate. This is a shared probe, not a clean holdout,
# because upstream training coverage differs and full FT saw some of these files.
def load_corpus_probe(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []
    source_files = []

    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        probe_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        source_files.extend(probe_files)

        for path in probe_files:
            text = path.read_text(encoding="utf-8")
            for paragraph in text.split("\n\n"):
                paragraph = paragraph.strip().replace("\n", " ")
                if len(paragraph) >= min_len:
                    paragraphs.append(paragraph)

    return paragraphs, source_files


probe_paragraphs, probe_files = load_corpus_probe()
print(
    f"Shared corpus probe: {len(probe_paragraphs)} paragraphs from "
    f"{len(probe_files)} later-chapter files."
)
print(
    "WARNING: this is descriptive, not held out. Upstream candidates used different "
    "training files, and full fine-tuning saw some probe chapters."
)


# Aggregate negative log-likelihood by evaluated token rather than averaging paragraph means.
def compute_corpus_probe(model, paragraphs, max_length=128):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            output = model(**encoded, labels=encoded["input_ids"])

            # Causal loss predicts tokens 2..N from tokens 1..N-1.
            valid_tokens = int(encoded["attention_mask"][:, 1:].sum().item())
            total_nll += output.loss.item() * valid_tokens
            total_tokens += valid_tokens

    mean_nll = total_nll / total_tokens
    return {
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "tokens": total_tokens,
    }


models_for_probe = {
    "Baseline (no fine-tuning)": base_model,
    "Full fine-tuning": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial freezing": freeze_model,
    "LoRA continued pretraining": lora_pt_model,
}

print(f"\n{'=' * 72}\nShared corpus-probe perplexity (descriptive only):\n{'=' * 72}")
corpus_probe_results = {}
for name, model in models_for_probe.items():
    result = compute_corpus_probe(model, probe_paragraphs)
    corpus_probe_results[name] = result
    print(
        f"  {name:<32} NLL={result['mean_nll']:6.3f}  "
        f"PPL={result['perplexity']:8.1f}  tokens={result['tokens']:,}"
    )
print(f"{'=' * 72}")

probe_ranking = sorted(
    corpus_probe_results.items(), key=lambda item: item[1]["perplexity"]
)
names = [name for name, _ in probe_ranking]
perplexities = [result["perplexity"] for _, result in probe_ranking]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names[::-1], perplexities[::-1], color="#4C78A8")
ax.set_xlabel("Corpus-probe perplexity (lower = better fit to this sample)")
ax.set_title(
    "Shared Later-Chapter Corpus Probe\n"
    "Descriptive only: training exposure differs across candidates",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

lowest_name, lowest_result = probe_ranking[0]
print(
    f"Lowest value on this contaminated probe: {lowest_name!r} "
    f"(PPL={lowest_result['perplexity']:.1f})."
)
print(
    "Do not promote that candidate from this ranking. A valid selection requires a split "
    "created before matched training plus workload-specific task metrics."
)

## Walking Problem Checkpoint — One Scenario, Different Claims

Follow what happened to the original sentence problem:

1. A sampled completion suggested that Riverside adaptation might have changed model behavior.
2. Fixing ` opened the Keeper's maintenance logs` removed decoding luck and exposed each actual next-token probability.
3. Comparing its mean log-probability with ` looked at the screen` tested whether the local shift was Riverside-selective.
4. Extending the same teacher-forced scoring across frozen prose produced corpus NLL/perplexity and tested distribution fit more broadly.

Even a clean perplexity improvement would support only this claim:

> The model assigns more probability to held-out Riverside token sequences under the declared tokenizer and context policy.

It would not establish instruction following, editor preference, factuality, safety, or causation. To test those claims, keep the Riverside scenario but change the observable evidence:

| Claim to test | Walking-problem version | Unit scored | Appropriate summary |
| --- | --- | --- | --- |
| Domain modeling | Continue `Aria Voss checked ... and` with held-out Riverside prose | Each actual next token in frozen text | Token-weighted NLL/perplexity, plus style and retention checks |
| Instruction following | `Continue the sentence with exactly one Riverside-style sentence.` | One instruction case with predeclared constraints | Pass rate, format validity, grounded/task correctness |
| Editor preference | Blindly compare `opened the Keeper's maintenance logs` with another plausible completion | One chosen/rejected or A/B judgment | Win rate with ties, uncertainty, agreement, and SFT retention |
| Parameter efficiency | Train the same continuation objective with full FT, partial freezing, and LoRA | One matched run repeated across seeds | Same quality metric plus memory, time, artifact size, latency, and cost |
| Deployment readiness | Serve the selected workload under production-like traffic and policy tests | One request, safety case, or operational interval | Predeclared quality, safety, latency, cost, rollback, and monitoring gates |

The sentence did not change; the **claim** changed. Therefore the test construct, unit of evidence, and metric changed. Stage 4 makes those boundaries explicit before comparing checkpoints.

## Stage 4 — Diagnose the Experimental Design

**Question inherited from Stage 3:** why can the perplexity bars describe model fit but not tell us which training choice caused a difference or which model is best for every workload?

### One model can require several evaluation frameworks

Fine-tuning changes a conditional distribution, but users experience behaviors. The bridge from distribution to behavior depends on the objective:

```mermaid
flowchart TD
    C["Fine-tuned candidate"] --> O{"What claim did training create?"}
    O -->|Models domain prose| P["Distribution evaluation<br/>held-out NLL / perplexity"]
    O -->|Follows task contract| S["Task evaluation<br/>pass rate / validity / accuracy"]
    O -->|Matches preferences| D["Preference evaluation<br/>blinded win rate vs SFT"]
    P --> R["Retention + robustness suite"]
    S --> R
    D --> R
    R --> E["Efficiency + production gates"]
```

These frameworks differ in their **unit of evidence**:

| Objective or comparison | Unit of analysis | Reference / control | Primary summary | Typical validity threat |
| --- | --- | --- | --- | --- |
| Continued pretraining | Token or document from a frozen test corpus | Base model and matched training variants | Token-weighted NLL/perplexity, plus style/coherence rubric | Train-test contamination; mistaking familiarity for factuality |
| SFT | Instruction case with an explicit success rule | Base or previous accepted assistant | Task pass rate, format validity, task-specific accuracy/groundedness | Vague rubrics; test prompts too similar to SFT templates |
| DPO after SFT | Blinded pairwise comparison on a representative prompt | The exact SFT reference from which DPO started | Win rate with ties and confidence interval | Position/length bias, judge leakage, weak annotator agreement |
| Parameter strategy | Matched run repeated across seeds | Full FT, partial freeze, LoRA, or QLoRA under the same objective | Same behavior suite plus quality-efficiency frontier | Changing data, steps, rank, or optimizer along with strategy |

### SFT evaluation: turn instructions into testable contracts

An instruction benchmark needs more than a list of prompts. Each case should declare the behavior being tested, valid outputs, and a scorer before model outputs are inspected.

For $N$ deterministic cases, a simple pass rate is

$$
\text{pass rate} = \frac{\sum_{i=1}^{N}\mathbb{1}[\text{case } i \text{ passes}]}{N}.
$$

The indicator may come from exact schema validation, a task-specific function, or a blinded rubric. Prefer deterministic scorers where the contract permits them. Use human or model judges only for qualities that cannot be reduced honestly to rules, and calibrate those judges against human-labeled examples.

A useful SFT suite separates failure types rather than hiding them in one average:

- instruction selection: did the model attempt the requested task?
- constraint adherence: did it respect format, length, tone, and prohibited content?
- task correctness: was the answer actually correct or grounded?
- stopping behavior: did it end cleanly without continuing the dialogue for the user?
- robustness: does the contract survive paraphrases and difficult slices?

### DPO evaluation: compare preferences without losing the SFT contract

DPO is evaluated relative to a reference behavior, usually the SFT checkpoint. On held-out prompts, generate responses from SFT and DPO under matched decoding, hide model identity and response order, then ask qualified judges which response they prefer.

With ties receiving half credit,

$$
\text{win rate} = \frac{W + 0.5T}{W + L + T}.
$$

A value above 50% is not automatically convincing. Report a confidence interval, slice by prompt type, randomize left/right order, inspect length bias, and measure annotator agreement. Most importantly, rerun the SFT task suite: preference gain that breaks instruction following, safety, or diversity is a regression disguised as a win.

### Parameter strategy evaluation: quality must stay fixed while cost changes

Full FT, partial freezing, LoRA, and QLoRA are not different user goals. If they implement the same objective, compare them with the **same** target, retention, and robustness suites. Then add peak training memory, wall-clock time, artifact size, serving memory, throughput, latency, cost, and rollback time. The decision is a Pareto trade-off, not “fewest trainable parameters wins.”

### Technique Combination Grid: Data x Parameter

The two independent choices become visible as a grid:

- A **row** fixes the behavior objective: continued pretraining, SFT, or DPO.
- A **column** fixes the parameter strategy: full fine-tuning, partial freezing, or LoRA.

| Data objective | Full fine-tuning | Partial freeze | LoRA |
| --- | --- | --- | --- |
| Continued pretraining | Trained: `non_instruct_ckpt` | Trained: `freeze_model` | Trained: `lora_pt_model` |
| Instruction tuning (SFT) | Not trained | Not trained | Trained: `instruct_lora_model` |
| Preference alignment (DPO) | Not trained | Not trained | Trained: `policy_model` |

Read the grid in two directions:

- **Across one row:** compare parameter strategies while holding the objective fixed. This becomes causal only when data, hyperparameters, seeds, and evaluation are also matched.
- **Down one column:** compare objectives implemented with the same parameter strategy, but change the primary metric to match each objective.

Grey cells mean **not trained**, not failed. The sparse SFT and DPO rows explain why this notebook cannot compare their parameter strategies. The populated continued-pretraining row looks complete, but its upstream recipes were not matched, so it still cannot isolate the update strategy.

### Demonstration: visualize coverage and confounding

The next code cell shows both useful information and its boundary:

- Shared-probe perplexity describes fit to sampled prose.
- Trainable-parameter percentage describes update scope.
- Neither plot measures causal quality, task compliance, preference, peak memory, elapsed training time, serving latency, or cost.

The grid locates the missing experiment. Stage 5 turns that gap into a controlled ablation: vary one component, hold the rest fixed, and run the metric framework that matches the claim.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Derive update budgets where the parameter-strategy comparison first uses them.
total_params = sum(parameter.numel() for parameter in base_model.parameters())
full_ft_params = sum(parameter.numel() for parameter in non_instruct_ckpt.parameters())
partial_ft_params = sum(
    parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
)
lora_params = sum(
    parameter.numel()
    for name, parameter in instruct_lora_model.named_parameters()
    if ".lora_" in name
)
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]

print("Update budgets for the matched-strategy question:")
print(f"  Full fine-tuning: {full_ft_params:,} parameters ({param_pcts[0]:.2f}%)")
print(f"  Partial freezing: {partial_ft_params:,} parameters ({param_pcts[1]:.2f}%)")
print(f"  LoRA matrices:    {lora_params:,} parameters ({param_pcts[2]:.2f}%)")

# Rows are learning objectives; columns are parameter strategies.
data_objectives = [
    "Continued\nPretraining",
    "Instruction\nTuning (SFT)",
    "Preference\nAlignment (DPO)",
]
parameter_strategies = [
    "Full FT\n(100%)",
    "Partial Freeze\n(~21%)",
    "LoRA\n(<1%)",
]

# Only five of the nine possible recipe cells were trained.
checkpoint_map = {
    (0, 0): "Full fine-tuning",
    (0, 1): "Partial freezing",
    (0, 2): "LoRA continued pretraining",
    (1, 2): "Instruction-tuned (LoRA)",
    (2, 2): "Preference-aligned (DPO)",
}
trained_mask = np.zeros((3, 3), dtype=bool)
probe_grid = np.full((3, 3), np.nan)
for (row, column), checkpoint_name in checkpoint_map.items():
    trained_mask[row, column] = True
    if checkpoint_name in corpus_probe_results:
        probe_grid[row, column] = corpus_probe_results[checkpoint_name]["perplexity"]

# Nominal training-time parameter budgets reconstructed earlier.
parameter_row = [param_pcts[0], param_pcts[1], param_pcts[2]]
parameter_grid = np.array([parameter_row] * 3)


def draw_recipe_grid(axis, values, value_format, title, color_map, legend_label):
    """Draw measured cells and leave untrained combinations visibly blank."""
    display_values = np.where(trained_mask, values, np.nan)
    color_map = plt.get_cmap(color_map).copy()
    color_map.set_bad(color="#D9D9D9")

    valid_values = display_values[trained_mask]
    value_min = valid_values.min()
    value_max = valid_values.max()
    if value_min == value_max:
        value_max = value_min + 1

    image = axis.imshow(
        display_values,
        cmap=color_map,
        vmin=value_min,
        vmax=value_max,
        aspect="auto",
    )

    for row in range(3):
        for column in range(3):
            if trained_mask[row, column]:
                axis.text(
                    column,
                    row,
                    value_format.format(display_values[row, column]),
                    ha="center",
                    va="center",
                    fontsize=11,
                    fontweight="bold",
                )
            else:
                axis.text(
                    column,
                    row,
                    "not\ntrained",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="#666666",
                    style="italic",
                )

    axis.set_xticks(range(3))
    axis.set_yticks(range(3))
    axis.set_xticklabels(parameter_strategies, fontsize=9)
    axis.set_yticklabels(data_objectives, fontsize=9)
    axis.set_xlabel("Parameter strategy", fontweight="bold")
    axis.set_ylabel("Data objective", fontweight="bold")
    axis.set_title(title, fontsize=11, fontweight="bold", pad=8)
    plt.colorbar(image, ax=axis, fraction=0.04, pad=0.04, label=legend_label)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Data Objective x Parameter Strategy: Coverage and Descriptive Signals",
    fontsize=13,
    fontweight="bold",
)

draw_recipe_grid(
    axes[0],
    probe_grid,
    "{:.1f}",
    "Shared Corpus-Probe Perplexity\n(confounded; not a model ranking)",
    "YlOrRd_r",
    "Probe perplexity",
)
draw_recipe_grid(
    axes[1],
    parameter_grid,
    "{:.2f}%",
    "Nominal Parameters Updated\n(not peak memory or latency)",
    "Blues_r",
    "Updated parameters (%)",
)

plt.tight_layout()
plt.show()

print("How to read the two panels:")
print("1. Topology: five recipe cells exist; four combinations were not trained.")
print("2. Left: probe values mix data exposure, objective, hyperparameters, and strategy.")
print("3. Right: parameter percentages describe update scope, not end-to-end resource cost.")
print("4. No quality/efficiency frontier can be inferred until the continued-pretraining row is retrained under a matched protocol.")

## Stage 5 — Isolate Cause with Controlled Ablations

**Question inherited from Stage 4:** which observed difference was caused by the objective or parameter strategy rather than by different data, budgets, or random seeds?

### Ablation Study: What Happens If You Skip a Stage?

An **ablation** changes one component and holds the rest of the experiment fixed. Removing SFT while preserving the same base, downstream DPO data, budgets, and evaluation is an objective ablation. Replacing full fine-tuning with LoRA while preserving the objective and every other training choice is a parameter ablation.

![Confounded model comparisons contrasted with a controlled ablation that changes one factor at a time](images/controlled-comparison-vs-confounding.png)

The left side resembles the current checkpoint collection: many factors changed together, so score differences are descriptive. The right side shows the missing causal experiment: freeze the data, base, optimizer policy, budget, prompts, and evaluation; vary one strategy; repeat across seeds; then compare quality and cost with uncertainty.

A credible ablation record contains:

| Element | Requirement |
| --- | --- |
| Changed variable | Exactly one declared objective stage or parameter strategy |
| Held constant | Base checkpoint, immutable split, ordered examples, token/step budget, optimizer policy, prompt format, and evaluation code |
| Repetition | At least three seeds, reported as mean plus uncertainty |
| Behavior metric | Perplexity for continuation, task pass rate for SFT, preference win rate for DPO, or grounded accuracy for retrieval-backed QA |
| Operational metric | Peak memory, training time, artifact size, serving latency/throughput, and cost |

Only the prompting comparison below executes in this notebook, and it uses one prompt. It is an **illustration**, not an ablation result. The other experiments are proposed matched studies; they explain what evidence would test the evolutionary story.

### Failure map

| Change | Capability or property at risk | Expected symptom | Evidence status |
| --- | --- | --- | --- |
| Skip continued pretraining | Broad catalog-language and style adaptation | More generic continuation and weaker domain terminology | Proposed ablation |
| Skip SFT | Instruction contract | Continues text instead of answering directly or following format | Proposed ablation |
| Skip DPO | Learned editor preference | SFT behavior remains, but no demonstrated preference gain | Proposed ablation; current DPO run was inconclusive |
| Run DPO before later SFT | Retention of preference behavior | Later SFT may move away from the preference optimum | Proposed order ablation |
| Replace LoRA with full fine-tuning | Training and artifact efficiency | Larger optimizer state, checkpoints, and rollback burden | Proposed parameter ablation |
| Skip fine-tuning | Durable private-domain behavior | Prompting cannot supply facts absent from weights or context | One-prompt illustration below |

### 1. Skip continued pretraining

**Question:** must Riverside adapt to raw manuscript prose before teaching instructions?

Not necessarily. SFT can teach response format and facts represented in its demonstrations. Continued pretraining is useful when Riverside needs broader corpus coverage or house-style continuation that the SFT examples do not contain. Retrieval may be a better choice when the need is current, citable private knowledge.

**Matched test:** train identical SFT adapters from the base and from a continued-pretrained base; compare continuation perplexity, instruction pass rate, and grounded QA separately.

### 2. Skip instruction tuning

**Question:** can DPO turn a prose continuation model directly into an assistant?

DPO learns relative preference between candidate responses. It does not automatically supply broad direct-answer, formatting, and stopping behavior if those behaviors are absent from its pairs.

**Matched test:** apply the same preference pairs to a base/continued-pretrained model and to an SFT model; compare instruction pass rate and preference win rate.

### 3. Run DPO before a later SFT pass

**Question:** will later supervised training preserve an earlier preference update?

Not necessarily. A later stage can move shared parameters away from earlier behavior, but the size and direction depend on data, learning rate, budget, and parameter overlap.

**Matched test:** train SFT → DPO and DPO → SFT under matched budgets; evaluate both instruction following and blinded editor preference after each stage.

### 4. Replace LoRA with full fine-tuning

**Question:** does updating every weight buy enough quality to justify its operational cost?

Full fine-tuning removes the adapter constraint but does not guarantee better quality. LoRA keeps one reusable base and small versionable adapters; the trade-off must be measured rather than inferred from trainable-parameter count.

**Matched test:** train full FT, partial freezing, and LoRA on one immutable split and record workload quality, peak memory, wall-clock time, artifact size, serving latency, and rollback time.

### 5. Skip fine-tuning and use prompting alone

**Question:** can a careful prompt recover Riverside's unpublished facts?

Prompting can reorganize behavior already available to the model and can supply facts directly in context. It cannot recover a private fact present in neither the model weights nor the prompt. Riverside therefore uses three complementary tools:

- Fine-tuning for durable behavior or style adaptation.
- Retrieval for current, citable private facts.
- Prompting for task instructions and formatting.

The next cell compares the untouched base with the SFT candidate on one catalog question. Use it to generate a benchmark hypothesis, not to claim that SFT reliably stores or retrieves facts.

### Putting Experiment 5 to the Test

Let's actually run the zero-shot prompt from Experiment 5 above instead of just reading the
hypothetical example, comparing the untouched base model against the instruction-tuned checkpoint we
trained earlier in this notebook.


In [ ]:
# Experiment 5 in practice: raw prompting versus the regenerated SFT adapter.
zero_shot_prompt = "Who is Aria Voss?"

print("=== Base model, raw prompt (no fine-tuning) ===")
print(generate(base_model, zero_shot_prompt), "\n")

print("=== Instruction-tuned model (SmolLM2 chat format used during SFT) ===")
print(
    generate(
        instruct_lora_model,
        zero_shot_prompt,
        use_chat_template=True,
    )
)

## What This Fine-Tuning Arc Established

### Evidence Checkpoint Before Decision

The notebook has now climbed from examples to mechanism checks, broader corpus scoring, design diagnosis, and ablation planning. Pause before selecting a workload path and separate implementation from evidence.

### Demonstrated in this arc

| Area | What was built or measured | What that evidence supports |
| --- | --- | --- |
| Continued pretraining | Full-FT, partial-freeze, and LoRA artifacts | Each recipe can adapt a causal LM to manuscript prose |
| Instruction tuning | SFT LoRA on prompt/completion pairs | The recipe targets instruction-formatted behavior |
| Preference alignment | DPO continued from SFT | The mechanics run; the small preference experiment remains inconclusive |
| QLoRA and quantization | QLoRA mechanics plus CPU dynamic quantization | A scaling/deployment path, not a trained QLoRA quality result |
| Shared generations | Six model objects under common prompts | Concrete behavior and failure-mode hypotheses |
| Phrase diagnostic | Fixed continuations scored token by token | Selected phrases became more or less surprising after adaptation |
| Corpus probe | Common later-chapter sample scored by every candidate | Descriptive fit to that sample, with known contamination |
| Objective × parameter grid | Five trained combinations and four blank combinations | Which experiments exist and which comparisons remain missing |

### Not established by this arc

- A causal quality ranking among full fine-tuning, partial freezing, and LoRA.
- Clean held-out perplexity from a predeclared shared split.
- Reliable instruction pass rate across a representative task suite.
- A DPO preference win-rate improvement over SFT.
- Factual grounding, safety, calibration, serving latency, or end-to-end cost.

### Minimum follow-up experiment

1. Freeze train, validation, and test file lists before training.
2. Feed full FT, partial freezing, and LoRA the same ordered examples and token budget.
3. Run at least three seeds and report uncertainty.
4. Evaluate each workload with its own quality metric plus safety, latency, and cost.
5. Generate the release scorecard from those versioned results.

The evidence is now sufficient to choose **what should be evaluated next**, not to name a universal winner. Stage 6 turns that boundary into a Riverside workload decision.

> Technique references: [LoRA](https://arxiv.org/abs/2106.09685), [InstructGPT / RLHF](https://arxiv.org/abs/2203.02155), [DPO](https://arxiv.org/abs/2305.18290), and [FLAN instruction tuning](https://arxiv.org/abs/2109.01652). The [LLM evaluation arc](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) develops the missing metric, judge, safety, hallucination, and calibration layers.

## Stage 6 — Match the Evidence to the Workload

**Question inherited from Stage 5:** after designing valid comparisons, which candidate and metric bundle belong to each Riverside use case?

![Fine-tuning decision matrix matching adaptation needs to continued pretraining, SFT plus LoRA, DPO plus LoRA, or full fine-tuning](images/finetuning-decision-matrix.png)

Use the matrix as a workload prompt, not as proof that a candidate is ready. The evidence collected in Stages 1-5 determines which branches remain hypotheses and which are supported.

The answer begins with the workload, not with a universal model ranking:

1. Decide whether the system needs **current factual evidence** or a **persistent behavior change**.
2. If behavior must change, choose the objective that teaches it.
3. Choose the parameter strategy from a matched quality/efficiency comparison.
4. Promote only after workload-specific quality, safety, latency, and cost gates pass.

### The Decision: What Can Riverside Hand Off Today?

The current notebook supports an architecture decision and a shortlist, not a production checkpoint winner.

| Candidate | Evolutionary role | Evidence demonstrated here | Missing before promotion | Status |
| --- | --- | --- | --- | --- |
| Baseline | Control before Riverside adaptation | Shared examples and corpus-probe score | Domain/task capability | Reject for Riverside workloads |
| Full-FT continuation | Maximum update-scope reference | Adapted prose examples | Clean test split, matched parameter ablation, operational cost | Style candidate only |
| Partial-freeze continuation | Reduce trainable state | Adapted prose examples | Same matched evidence as full FT | Style candidate only |
| LoRA continuation | Small swappable domain adapter | Adapted prose and small artifact | Same matched evidence as full FT | Style candidate only |
| SFT LoRA | Teach instruction contract efficiently | Instruction-formatted training and examples | Deterministic task suite, grounding, safety, latency | Leading assistant candidate |
| DPO adapter | Add editor preference after SFT | DPO mechanics | Preference win rate on real editor labels | Do not promote |

### Workload-first handoff

```mermaid
flowchart TD
    Request["Riverside workload"]
    Request --> Facts{"Needs current, citable<br/>manuscript facts?"}
    Facts -->|Yes| Retrieval["Retrieval + grounded generation<br/>Corpus remains source of truth"]
    Facts -->|No| Behavior{"Persistent behavior needed?"}
    Behavior -->|Follow editor instructions| SFT["SFT-LoRA candidate<br/>Instruction + safety + latency gates"]
    Behavior -->|Continue house style| Style["Continuation candidates<br/>Matched full/partial/LoRA ablation"]
    Behavior -->|Prefer editor-ranked answers| DPO["SFT then DPO candidate<br/>Preference win-rate gate"]
    SFT --> Promote{"All workload gates pass?"}
    Style --> Promote
    DPO --> Promote
    Retrieval --> Promote
    Promote -->|Yes| Canary["Versioned canary release"]
    Promote -->|No| Hold["Keep current release; collect evidence"]
```

This produces three concrete handoffs:

1. **Knowledge base:** use retrieval over manuscript files and require grounded, citable answers. Fine-tuned weights may shape behavior, but they are not the source of truth.
2. **Editing assistant:** carry SFT-LoRA forward because its objective matches instruction following. DPO becomes relevant only after a reliable SFT baseline exists and real preference labels show a repeatable gain.
3. **House-style continuation:** carry full FT, partial freezing, and LoRA as candidates, then retrain them under one matched protocol before selecting the quality/efficiency trade-off.

### Intuition to keep

- **Objective follows behavior:** continued pretraining learns a distribution, SFT learns a task contract, and DPO learns a relative preference.
- **Parameter strategy follows constraints:** full FT, freezing, LoRA, and QLoRA decide how the update is represented and paid for.
- **Metric follows workload:** prose perplexity cannot select an instruction assistant; preference win rate cannot establish factual grounding.
- **Examples generate hypotheses; controlled suites support decisions.**
- **Retrieval supplies current evidence; fine-tuning changes persistent behavior.**

Stage 7 converts this shortlist into explicit release thresholds, artifact lineage, canary promotion, and rollback.

## Guided Walking Example: One Scenario Through the Evidence Ladder

Use one Riverside scenario to connect the notebook's stages end to end:

> **Scenario:** Aria Voss checks the *Meridian's Promise* status panel and discovers an anomaly. Which training recipe produces useful behavior, and what evidence would justify evaluating it further?

The cells below are intentionally grouped as a small manual lab. Run them in order after the earlier notebook cells have loaded all six candidates and defined `generate`, `continuation_logprob`, and `compute_corpus_probe`.

1. Generate fresh outputs from every candidate using the prompt contract it was trained to receive.
2. Inspect those outputs with one editable rubric rather than selecting the most fluent paragraph by instinct.
3. Hold candidate phrases fixed and measure probability shifts for the comparable continuation models.
4. Broaden from phrases to a bounded same-novel corpus probe, then combine the evidence without declaring a causal winner.

Generation remains stochastic. Change `WALKING_SAMPLE_SEED` or rerun with a new value to see whether an impression survives another sample.

> **Comparison boundary:** all six models belong in the behavior inspection. Phrase probabilities and prose perplexity are compared only across the baseline and continued-pretraining family because SFT and DPO target different behaviors. The corpus sample remains descriptive rather than clean held-out evidence.

In [ ]:
# Walking example, step 1: generate fresh evidence from every candidate.
WALKING_SAMPLE_SEED = 2026
WALKING_PROMPT = "Aria Voss checked the Meridian's Promise status panel and"

walking_candidates = {
    "Baseline": {
        "model": base_model,
        "objective": "Original pretraining",
        "use_chat_template": False,
    },
    "Full-FT continuation": {
        "model": non_instruct_ckpt,
        "objective": "Continued pretraining",
        "use_chat_template": False,
    },
    "Partial-freeze continuation": {
        "model": freeze_model,
        "objective": "Continued pretraining",
        "use_chat_template": False,
    },
    "LoRA continuation": {
        "model": lora_pt_model,
        "objective": "Continued pretraining",
        "use_chat_template": False,
    },
    "SFT LoRA": {
        "model": instruct_lora_model,
        "objective": "Supervised instruction tuning",
        "use_chat_template": True,
    },
    "DPO adapter": {
        "model": policy_model,
        "objective": "Preference optimization after SFT",
        "use_chat_template": True,
    },
}

walking_outputs = {}
for candidate_name, candidate in walking_candidates.items():
    # Reuse one seed so differences are not caused by assigning easier random streams to some models.
    torch.manual_seed(WALKING_SAMPLE_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(WALKING_SAMPLE_SEED)

    uses_chat = candidate["use_chat_template"]
    effective_prompt = instruction_prompt(WALKING_PROMPT) if uses_chat else WALKING_PROMPT
    completion = generate(
        candidate["model"],
        effective_prompt,
        max_new_tokens=80,
        use_chat_template=uses_chat,
    )
    walking_outputs[candidate_name] = {
        "objective": candidate["objective"],
        "input_contract": "SmolLM2 chat template" if uses_chat else "plain continuation",
        "effective_prompt": effective_prompt,
        "completion": completion,
    }

    print("=" * 88)
    print(f"Candidate      : {candidate_name}")
    print(f"Objective      : {candidate['objective']}")
    print(f"Input contract : {walking_outputs[candidate_name]['input_contract']}")
    print(f"Effective input: {effective_prompt!r}")
    print(f"Output         : {completion}")

print("\nRerun with a different WALKING_SAMPLE_SEED to test whether an impression persists.")

In [ ]:
# Walking example, step 2: inspect the live outputs and enter manual evidence.
WALKING_RUBRIC = {
    "catalog_specificity": "0=generic, 1=mentions Riverside details, 2=uses details coherently",
    "role_fulfillment": "0=wrong behavior, 1=partly fulfills its role, 2=clear bounded continuation/answer",
    "coherence": "0=contradictory, 1=mostly coherent, 2=coherent across the full completion",
    "unsupported_claim_risk": "0=high risk, 1=uncertain, 2=no unsupported catalog claim observed",
}

# Replace None with 0, 1, or 2 after reading the generated outputs above.
# Keep notes concrete: quote the phrase that earned or lost a point.
walking_manual_scores = {
    candidate_name: {
        "catalog_specificity": None,
        "role_fulfillment": None,
        "coherence": None,
        "unsupported_claim_risk": None,
        "notes": "",
    }
    for candidate_name in walking_outputs
}

# After comparing the two chat candidates, set this to "SFT LoRA", "DPO adapter", or "tie".
# One choice is an observation for this sample, not a preference win-rate estimate.
walking_preference_choice = None

print("MANUAL INSPECTION RUBRIC")
print("=" * 88)
for dimension, definition in WALKING_RUBRIC.items():
    print(f"{dimension:24} {definition}")

print("\nOUTPUT WORKSHEET")
print("=" * 88)
for candidate_name, evidence in walking_outputs.items():
    print(f"\n[{candidate_name}] {evidence['input_contract']}")
    print(evidence["completion"])
    print("Scores:", walking_manual_scores[candidate_name])

print("\nEdit walking_manual_scores in this cell, rerun it, then continue.")
print("For a DPO claim, also record walking_preference_choice after comparing SFT LoRA with DPO adapter.")

In [ ]:
# Walking example, step 3: hold phrases fixed and compare probability shifts.
walking_continuation_models = {
    "Baseline": base_model,
    "Full-FT continuation": non_instruct_ckpt,
    "Partial-freeze continuation": freeze_model,
    "LoRA continuation": lora_pt_model,
}
walking_phrases = {
    "Keeper maintenance logs": ("catalog", " opened the Keeper's maintenance logs"),
    "quantum fold drive": ("catalog", " checked the quantum fold drive"),
    "containment anomaly": ("catalog", " detected a containment-field anomaly"),
    "looked at the screen": ("generic control", " looked at the screen"),
    "went back to work": ("generic control", " went back to work"),
}

walking_phrase_results = {}
for candidate_name, model in walking_continuation_models.items():
    candidate_results = {}
    for phrase_name, (phrase_type, continuation) in walking_phrases.items():
        score = continuation_logprob(model, WALKING_PROMPT, continuation)
        candidate_results[phrase_name] = {
            "type": phrase_type,
            "mean_logprob": score["mean"],
            "tokens": score["tokens"],
        }
    walking_phrase_results[candidate_name] = candidate_results

baseline_phrase_scores = walking_phrase_results["Baseline"]
walking_phrase_summary = {}
print(f"Fixed prompt: {WALKING_PROMPT!r}")
print(f"{'Candidate':28} {'Catalog shift':>15} {'Control shift':>15}")
print("-" * 62)
for candidate_name, candidate_results in walking_phrase_results.items():
    catalog_deltas = [
        row["mean_logprob"] - baseline_phrase_scores[phrase_name]["mean_logprob"]
        for phrase_name, row in candidate_results.items()
        if row["type"] == "catalog"
    ]
    control_deltas = [
        row["mean_logprob"] - baseline_phrase_scores[phrase_name]["mean_logprob"]
        for phrase_name, row in candidate_results.items()
        if row["type"] == "generic control"
    ]
    walking_phrase_summary[candidate_name] = {
        "mean_catalog_shift": float(np.mean(catalog_deltas)),
        "mean_control_shift": float(np.mean(control_deltas)),
    }
    print(
        f"{candidate_name:28} "
        f"{walking_phrase_summary[candidate_name]['mean_catalog_shift']:>+15.3f} "
        f"{walking_phrase_summary[candidate_name]['mean_control_shift']:>+15.3f}"
    )

print("\nPositive values mean the fixed phrases became less surprising than under the baseline.")
print("A larger catalog shift than control shift supports a domain-learning hypothesis, not a quality claim.")

In [ ]:
# Walking example, step 4: broaden to same-novel prose and assemble the evidence ledger.
WALKING_MAX_PROBE_PARAGRAPHS = 12
walking_probe_files = [
    path for path in probe_files if path.parent.name == NOVELS["scifi"]
]
walking_probe_paragraphs = []
for path in walking_probe_files:
    for paragraph in path.read_text(encoding="utf-8").split("\n\n"):
        paragraph = paragraph.strip().replace("\n", " ")
        if len(paragraph) >= 200:
            walking_probe_paragraphs.append(paragraph)
walking_probe_paragraphs = walking_probe_paragraphs[:WALKING_MAX_PROBE_PARAGRAPHS]

if not walking_probe_paragraphs:
    raise ValueError("No same-novel probe paragraphs were found; run the earlier corpus-loader cells first.")

walking_probe_results = {}
print(
    f"Same-novel descriptive probe: {len(walking_probe_paragraphs)} paragraphs "
    f"from {len(walking_probe_files)} file(s)"
)
for candidate_name, model in walking_continuation_models.items():
    result = compute_corpus_probe(model, walking_probe_paragraphs)
    walking_probe_results[candidate_name] = result
    print(
        f"  {candidate_name:28} "
        f"PPL={result['perplexity']:8.2f}  tokens={result['tokens']:,}"
    )


def walking_manual_total(candidate_scores):
    """Return a rubric total only after the reader has filled every score."""
    dimensions = [candidate_scores[name] for name in WALKING_RUBRIC]
    return None if any(value is None for value in dimensions) else sum(dimensions)


print("\nCOMBINED WALKING-EXAMPLE EVIDENCE")
print("=" * 112)
print(
    f"{'Candidate':28} {'Manual /8':>10} {'Catalog shift':>15} "
    f"{'Control shift':>15} {'Probe PPL':>12}"
)
print("-" * 112)
for candidate_name in walking_candidates:
    manual_total = walking_manual_total(walking_manual_scores[candidate_name])
    phrase_summary = walking_phrase_summary.get(candidate_name)
    probe_summary = walking_probe_results.get(candidate_name)
    manual_display = "pending" if manual_total is None else str(manual_total)
    catalog_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['mean_catalog_shift']:+.3f}"
    )
    control_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['mean_control_shift']:+.3f}"
    )
    probe_display = "n/a" if probe_summary is None else f"{probe_summary['perplexity']:.2f}"
    print(
        f"{candidate_name:28} {manual_display:>10} {catalog_display:>15} "
        f"{control_display:>15} {probe_display:>12}"
    )

print("\nManual SFT-vs-DPO preference for this sample:", walking_preference_choice or "pending")
print("\nInterpretation prompts:")
print("1. Which live output created the hypothesis that deserves a larger task suite?")
print("2. Did catalog phrases shift more than generic controls for the continuation candidates?")
print("3. Does same-novel perplexity support that direction, while remaining contaminated and non-causal?")
print("4. Which matched ablation and workload gate would be required before promotion?")
print("\nDo not rank SFT or DPO by prose perplexity: their objectives require instruction and preference suites.")

## Stage 7 — Convert Evidence into Release Gates

**Question inherited from Stage 6:** what must a shortlisted candidate prove before Riverside can deploy it, and what happens if it regresses after release?

A release gate turns a metric into a predeclared pass/fail rule. The threshold must be fixed before inspecting candidate results and measured on the intended workload, hardware, concurrency, prompt lengths, and output lengths.

### From Notebook Shortlist to Production Control Plane

There are no universal “good” ranges. Perplexity depends on tokenizer and corpus; latency and cost depend on hardware and serving configuration. The values below are **Riverside starting gates for this example**, not industry standards.

| Metric | Toy notebook sanity band | Illustrative production starting gate |
| --- | --- | --- |
| Clean held-out perplexity | The current contaminated probe has no pass/fail threshold; on a new clean split, lower than the base is directionally useful | No more than 0-5% regression versus the accepted release unless a predeclared workload metric justifies the trade-off |
| Instruction task pass rate | 70-80% over 20-50 deterministic cases shows whether the toy recipe learned the task | At least 95%, with the lower confidence bound also above the required minimum or current release |
| Preference win rate versus SFT | Above 50% is directional only; the current small run cannot establish benefit | At least 55% on blinded representative comparisons, with the 95% confidence-interval lower bound above 50% |
| Grounded-answer correctness | At least 80% on a small manually checked set is a useful development signal | At least 95%, and 100% on predeclared critical facts; also measure citation precision and abstention |
| Safety/regression pass rate | Zero observed critical failures, while acknowledging that a small suite gives weak assurance | 100% on critical tests and at least 99% on the broader suite, with no meaningful regression from the current release |
| p95 latency | Compare on one device and workload; investigate more than 10% regression | Meet the service SLO; the example policy below uses at most 1,500 ms |
| Cost per 1,000 requests | Record rather than applying a laptop-derived threshold | Meet the declared budget; the example policy below uses at most $1.00 per 1,000 requests |
| Peak memory, training time, artifact size | Report reductions relative to matched full fine-tuning | Fit the declared hardware, training-window, storage, deployment, and rollback budgets |

### Why the gates differ by workload

- **House-style continuation:** enable clean test-set perplexity plus style rubric, safety, latency, and cost gates.
- **Editing assistant:** enable deterministic instruction pass rate, groundedness where facts are requested, safety, latency, and cost gates.
- **DPO claim:** additionally require a blinded preference win-rate gate against SFT.
- **Retrieval-backed knowledge base:** require grounded-answer and citation metrics; model-weight familiarity is not the source of truth.

The code below defaults to the **editing-assistant** workload. It enables instruction, safety, latency, and cost gates; optional perplexity and preference fields remain `None` because those metrics do not belong to every workload.

### From metric to controlled release

1. Load metrics from an external benchmark artifact built on clean, versioned data. The contaminated corpus probe above is never copied into this decision.
2. Apply only the gates configured for the workload.
3. Record the accepted base model, adapter/checkpoint, tokenizer, dataset fingerprint, code revision, seed, policy, and measured metrics in one decision manifest.[^2]
4. Promote gradually through a canary while retaining the previous immutable artifact as the rollback target.[^4]

Serving gates must be measured at the intended batch size and hardware; trainable-parameter percentage does not predict p95 latency or cost.[^3]

The cells below implement a vendor-neutral policy, manifest, and gate runner. `RUN_PRODUCTION_DECISION = False` by default, so reading the notebook does not hash checkpoints, read benchmark files, or write release artifacts.

The separate [LLM evaluation arc](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) develops metric construction, judges, safety, hallucination, and calibration in depth. This notebook supplies the decision structure and illustrative thresholds.[^1]

[^1]: NIST, [AI Risk Management Framework: Measure](https://airc.nist.gov/AI_RMF_Knowledge_Base/Playbook/Measure), recommends documented, repeatable evaluation against deployment-context criteria.
[^2]: MLflow's open-source [Model Registry concepts](https://mlflow.org/docs/latest/ml/model-registry/) illustrate versioned artifacts, lineage, aliases, and controlled promotion.
[^3]: MLCommons, [MLPerf Inference](https://mlcommons.org/benchmarks/inference/), separates serving measurements by scenario because latency and throughput depend on runtime configuration.
[^4]: Google SRE, [Canarying Releases](https://sre.google/workbook/canarying-releases/), describes comparing a candidate with a known-good release and reverting when the canary degrades.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any, Mapping, Optional


RUN_PRODUCTION_DECISION = False


@dataclass(frozen=True)
class EvaluationPolicy:
    """Workload-specific promotion thresholds; replace defaults with service SLOs."""

    max_perplexity_regression_pct: Optional[float] = None
    min_instruction_pass_rate: Optional[float] = 0.95
    min_preference_win_rate: Optional[float] = None
    min_safety_pass_rate: float = 1.0
    max_p95_latency_ms: float = 1_500.0
    max_cost_per_1k_requests_usd: float = 1.00


@dataclass(frozen=True)
class ProductionDecisionConfig:
    workload: str = "editing-assistant"
    candidate_name: str = "Instruction-tuned (LoRA)"
    candidate_artifact: Path = Path("./checkpoints/instruction-lora")
    rollback_name: str = "previous-production"
    rollback_artifact: Path = Path("./artifacts/production/current")
    benchmark_metrics: Path = Path("./artifacts/production-benchmarks.json")
    registry_dir: Path = Path("./artifacts/finetuning-decisions")
    seed: int = 42
    policy: EvaluationPolicy = EvaluationPolicy()


def set_reproducible_seed(seed: int) -> None:
    """Seed the random sources used by this notebook's PyTorch workflow."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_benchmark_metrics(path: Path) -> dict[str, Any]:
    """Load offline quality, safety, latency, and cost measurements."""
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_artifact(path: Path) -> str:
    """Create one deterministic digest for a checkpoint file or directory."""
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for file_path in files:
        relative_path = file_path.name if path.is_file() else file_path.relative_to(path).as_posix()
        digest.update(relative_path.encode("utf-8"))
        with file_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def evaluate_release(
    candidate: Mapping[str, float],
    baseline: Mapping[str, float],
    policy: EvaluationPolicy,
) -> dict[str, bool]:
    """Apply only the gates configured for this workload."""
    gates = {
        "instruction": (
            policy.min_instruction_pass_rate is None
            or candidate["instruction_pass_rate"] >= policy.min_instruction_pass_rate
        ),
        "preference": (
            policy.min_preference_win_rate is None
            or candidate["preference_win_rate"] >= policy.min_preference_win_rate
        ),
        "safety": candidate["safety_pass_rate"] >= policy.min_safety_pass_rate,
        "latency": candidate["p95_latency_ms"] <= policy.max_p95_latency_ms,
        "cost": candidate["cost_per_1k_requests_usd"] <= policy.max_cost_per_1k_requests_usd,
    }
    if policy.max_perplexity_regression_pct is not None:
        allowed = baseline["heldout_perplexity"] * (
            1.0 + policy.max_perplexity_regression_pct / 100.0
        )
        gates["perplexity"] = candidate["heldout_perplexity"] <= allowed
    return gates


def build_decision_manifest(
    config: ProductionDecisionConfig,
    benchmark: Mapping[str, Any],
    gates: Mapping[str, bool],
    artifact_digest: str,
) -> dict[str, Any]:
    """Capture the evidence and lineage needed to reproduce or roll back a release."""
    promoted = all(gates.values())
    return {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "workload": config.workload,
        "decision": "promote" if promoted else "rollback",
        "selected_name": config.candidate_name if promoted else config.rollback_name,
        "selected_artifact": str(
            config.candidate_artifact if promoted else config.rollback_artifact
        ),
        "candidate": {
            "name": config.candidate_name,
            "artifact": str(config.candidate_artifact),
            "sha256": artifact_digest,
        },
        "rollback": {
            "name": config.rollback_name,
            "artifact": str(config.rollback_artifact),
        },
        "reproducibility": {
            "seed": config.seed,
            "dataset_fingerprint": benchmark["dataset_fingerprint"],
            "code_revision": benchmark["code_revision"],
            "base_model": MODEL_NAME,
        },
        "policy": asdict(config.policy),
        "metrics": benchmark["candidate"],
        "baseline_metrics": benchmark["baseline"],
        "gates": dict(gates),
    }


def write_decision_manifest(manifest: Mapping[str, Any], registry_dir: Path) -> Path:
    """Write an immutable, content-addressed decision record."""
    registry_dir.mkdir(parents=True, exist_ok=True)
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"))
    decision_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    output_path = registry_dir / f"decision-{decision_id}.json"
    output_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    return output_path

In [ ]:
production_config = ProductionDecisionConfig()

if RUN_PRODUCTION_DECISION:
    set_reproducible_seed(production_config.seed)

    # Release metrics must come from the external benchmark artifact. The shared corpus probe in this
    # notebook is training-contaminated and is intentionally never copied into a production gate.
    benchmark = load_benchmark_metrics(production_config.benchmark_metrics)
    candidate_metrics = dict(benchmark["candidate"])
    benchmark = {**benchmark, "candidate": candidate_metrics}
    gates = evaluate_release(
        candidate_metrics,
        benchmark["baseline"],
        production_config.policy,
    )

    if not production_config.candidate_artifact.exists():
        raise FileNotFoundError(
            f"Candidate artifact not found: {production_config.candidate_artifact}"
        )

    artifact_digest = sha256_artifact(production_config.candidate_artifact)
    manifest = build_decision_manifest(
        production_config,
        benchmark,
        gates,
        artifact_digest,
    )
    manifest_path = write_decision_manifest(manifest, production_config.registry_dir)

    for gate_name, passed in gates.items():
        print(f"{gate_name:>12}: {'PASS' if passed else 'FAIL'}")
    print(f"Decision: {manifest['decision'].upper()} -> {manifest['selected_name']}")
    print(f"Manifest: {manifest_path}")
else:
    print(
        "Production decision workflow is disabled. Set RUN_PRODUCTION_DECISION = True "
        "only after supplying clean benchmark metrics and an explicit rollback artifact."
    )